# 07 · Pipelines with Ray

*The phase 1 ingestion rebuilt as a pipeline that can run on one laptop or a cluster.*

Phase 1 built the station data with a thread pool and a few conventions: one unit of work per station-year, raw files never touched, outputs written atomically. That was enough for 165 stations. It isn't a pipeline yet. Nothing records what was done or with which code, a backfill and next month's update are different scripts and the only way to scale is a bigger laptop.

This notebook rebuilds it with **Ray**. Ray is a Python framework for running functions and stateful objects across many cores or machines. It was built at Berkeley's RISELab for workloads that don't fit neatly into SQL-style batch jobs (like reinforcement learning, hyperparameter search and model serving) and it has become a common backbone for ML data pipelines.

The goal is to learn Ray's model by using it on a real job. Three ideas carry most of it:

- A **task** is a function run somewhere in the cluster. You call it and get back a reference to its result and not the result.
- An **object reference** is a handle to a value in Ray's shared **object store**. Passing references between tasks lets Ray move data only when and where it's needed.
- An **actor** is a stateful worker: a Python object living in its own process whose methods run as tasks. It's how you keep something expensive (like an open connection) alive between calls.

On top of those sits **Ray Data** (a library for streaming datasets through a chain of transformations) which I use at the end.

**Read first:** Ray's [Key concepts](https://docs.ray.io/en/latest/ray-core/key-concepts.html), [Tasks](https://docs.ray.io/en/latest/ray-core/tasks.html), [Actors](https://docs.ray.io/en/latest/ray-core/actors.html), [Fault tolerance](https://docs.ray.io/en/latest/ray-core/fault-tolerance.html) and [Ray Data key concepts](https://docs.ray.io/en/latest/data/key-concepts.html). About 45 minutes.

In [1]:
# Setup: identical in every notebook. Data lives in ../data, which git ignores.
import os
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
os.makedirs(DATA_DIR, exist_ok=True)
print("Data directory:", DATA_DIR)

Data directory: /Users/ashu/Workspace/weather-data-lab/data


In [2]:
import warnings; warnings.filterwarnings("ignore")
import time
import pandas as pd
from pathlib import Path
import ray
from wxlab import stations as st, pipeline as pl

RAW = Path(DATA_DIR) / "raw" / "isd"
OUT = Path(DATA_DIR) / "work" / "ray"                 # a separate output tree, so phase 1's data is untouched
STD, MANIFEST = OUT / "standardized", OUT / "manifest.parquet"
station_ids = sorted(p.stem for p in (Path(DATA_DIR) / "standardized" / "isd" / "year=2023").glob("*.parquet"))
import logging
ray.init(num_cpus=4, ignore_reinit_error=True, include_dashboard=False, logging_level=logging.ERROR, log_to_driver=False)
ray.data.DataContext.get_current().enable_progress_bars = False
logging.getLogger("ray.data").setLevel(logging.ERROR)
print(len(station_ids), "stations;", ray.available_resources().get("CPU"), "CPUs for Ray")

2026-11-09 11:23:24,426	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


164 stations; 4.0 CPUs for Ray


## Step 1 · Units, a manifest and a plan

The framework-neutral parts live in `wxlab.pipeline`. A **unit** is one station-year. The **manifest** is a table with a row per attempted unit: its key, a fingerprint of its input file, the code version, the outcome and the output path. **Planning** compares the units I want with the manifest and returns only what's missing, failed or out of date because its input or the code changed.

That last part is what makes reprocessing safe. Bump `CODE_VERSION` and every unit is planned again. Re-download one raw file and only its unit is.

In [3]:
wanted = pl.units_for(station_ids, range(2021, 2025))
manifest = pl.load_manifest(MANIFEST)
todo = pl.plan(wanted, manifest, RAW)
print(f"{len(wanted)} units wanted, {len(manifest)} manifest rows, {len(todo)} to run")

656 units wanted, 0 manifest rows, 653 to run


## Step 2 · The backfill as Ray tasks

`pl.run_unit` is a plain function with no Ray in it. `ray.remote` turns it into a task. Calling `.remote()` returns an object reference immediately and Ray schedules the work on a free CPU. `ray.wait` hands back results as they finish rather than in submission order so the manifest can be written in batches as work completes. If the notebook dies halfway the next run's plan picks up exactly what's left.

`max_retries` tells Ray to rerun a task if its worker process dies which is the cheap kind of fault tolerance. It's safe only because the unit is idempotent: running it twice gives the same file.

In [4]:
run_unit = ray.remote(max_retries=2)(pl.run_unit)

t0 = time.time()
pending = [run_unit.remote(u, RAW, STD) for u in todo]
rows = []
while pending:
    done, pending = ray.wait(pending, num_returns=min(32, len(pending)), timeout=30)
    rows += ray.get(done)
    if len(rows) % 128 < 32:
        manifest = pl.record(MANIFEST, rows); rows = []
manifest = pl.record(MANIFEST, rows)
print(f"{len(todo)} units in {time.time() - t0:.0f} s")
manifest.groupby("status").agg(units=("key", "size"), rows=("rows", "sum"), seconds=("seconds", "sum"))

653 units in 21 s


,units,rows,seconds
status,,,
ok,653,5248979,81.976


The tasks did about 82 seconds of work in 21 seconds of wall time (a speed-up of about 4 on 4 CPUs). (Four leaves memory for everything else on a 16 GB laptop. Each Ray worker is a full Python process.) Three station-years have no raw file and were never planned.

Run the plan again and it's empty. Nothing is redone because nothing changed.

In [5]:
print(len(pl.plan(wanted, pl.load_manifest(MANIFEST), RAW)), "units to run")

0 units to run


How does that compare with phase 1's thread pool? Parsing a station-year is CPU-bound pandas work. Python threads share one interpreter lock so they mostly take turns. Ray's workers are separate processes. Same 120 units, same number of workers, written to a scratch folder:

In [6]:
from concurrent.futures import ThreadPoolExecutor
sample, scratch = wanted[:120], OUT / "scratch"
t0 = time.time()
with ThreadPoolExecutor(4) as pool:
    list(pool.map(lambda u: pl.run_unit(u, RAW, scratch / "threads"), sample))
t_threads = time.time() - t0
t0 = time.time()
ray.get([run_unit.remote(u, RAW, scratch / "ray") for u in sample])
t_ray = time.time() - t0
print(f"threads: {t_threads:.0f} s   Ray tasks: {t_ray:.0f} s   ({t_threads / t_ray:.1f}x)")

threads: 10 s   Ray tasks: 3 s   (2.9x)


## Step 3 · Incremental runs use the same code

A live pipeline gets one new piece at a time. To simulate it without touching the raw files (they stay immutable) I make the manifest forget 2024 as if that year had just arrived. The plan finds exactly those units and the same task runs them. No second script.

In [7]:
m = pl.load_manifest(MANIFEST)
m[m.year != 2024].to_parquet(MANIFEST, index=False)   # pretend 2024 is new
todo = pl.plan(wanted, pl.load_manifest(MANIFEST), RAW)
print(len(todo), "units planned; years:", sorted({u.year for u in todo}))
manifest = pl.record(MANIFEST, ray.get([run_unit.remote(u, RAW, STD) for u in todo]))

164 units planned; years: [2024]


For a real live feed the trigger would be an event rather than a loop. NOAA publishes a notification for every new object in its buckets so a small listener can turn "new file landed" into "plan this unit". The unit, the plan and the manifest don't change.

## Step 4 · Actors for state that should live between calls

Phase 1 learned that opening a new HTTPS connection for every request failed at scale and one reused connection per worker fixed it. In Ray that's an **actor**: a worker process that holds state. A small pool of downloader actors (each with its own session) does what the thread-local sessions did and it works the same way on a cluster.

In [8]:
@ray.remote
class Downloader:
    def __init__(self):
        import requests
        self.session = requests.Session()               # opened once per actor and reused

    def head(self, station_id, year):
        url = f"https://{st.BUCKET}.s3.amazonaws.com/{year}/{station_id}.csv"
        r = self.session.head(url, timeout=30)
        return station_id, year, r.status_code, int(r.headers.get("Content-Length", 0))

pool = ray.util.ActorPool([Downloader.remote() for _ in range(4)])
sample = [(s, 2024) for s in station_ids[:40]]
sizes = list(pool.map_unordered(lambda a, v: a.head.remote(*v), sample))
pd.DataFrame(sizes, columns=["station_id", "year", "status", "bytes"]).describe()

,year,status,bytes
count,40.0,40.0,4.000000e+01
mean,2024.0,200.0,6.375905e+06
std,0.0,0.0,2.752139e+06
min,2024.0,200.0,3.355850e+05
25%,2024.0,200.0,4.983914e+06
50%,2024.0,200.0,5.936686e+06
75%,2024.0,200.0,7.961490e+06
max,2024.0,200.0,1.164710e+07


Only `HEAD` requests here so nothing is downloaded again. The shape is the point: the session is opened once per actor and lives as long as the actor.

## Step 5 · Ray Data for the next stage

Tasks are good for "run this function on each unit". The QA stage is different. It reads many units and groups them differently (by month across all stations) because the neighbour checks need every station's data for the same hours. **Ray Data** handles that kind of whole-dataset step. It reads files as a stream of blocks, applies transformations batch by batch and uses backpressure so a slow stage doesn't let the fast ones fill up memory.

As a small example let's compute each station's monthly completeness and mean temperature straight from the standardized files.

In [9]:
ds = ray.data.read_parquet(str(STD), columns=["station_id", "time", "air_temperature"])

def monthly(batch: pd.DataFrame) -> pd.DataFrame:
    batch["month"] = batch.time.dt.strftime("%Y-%m")
    return batch.groupby(["station_id", "month"]).agg(hours=("air_temperature", "count"),
                                                      mean_t=("air_temperature", "mean")).reset_index()

summary = ds.map_batches(monthly, batch_format="pandas").to_pandas()
summary = summary.groupby(["station_id", "month"]).agg(hours=("hours", "sum")).reset_index()  # blocks can split a month
summary.describe()

(raylet) WARNING: 16 PYTHON worker processes have been started on node: 8da22e3e5785f5246652d2ec582f6df17bc64e0a87bafc2a402269c9 with address: 127.0.0.1, which is 4x the maximum expected startup concurrency (4). This could be a result of using a large number of actors, tasks blocked in ray.get() calls, or tasks with fractional CPU requests (e.g., num_cpus=0.1) allowing high concurrency. See https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds.


,hours
count,7652.0
mean,666.381077
std,154.808804
min,0.0
25%,678.0
50%,720.0
75%,743.0
max,744.0


Notice the second `groupby`. A block can hold part of a station-month so per-block results have to be combined again. That's a small taste of the **shuffle** problem: any step that groups by a key the data isn't already partitioned by needs data moved between workers. Ray Data has `groupby` for this but large shuffles are where Spark's engine is more mature. Notebook 08 shows the same step there.

## Where Ray fits

These are the documented strengths and limits along with what this job showed.

- **Good at:** scaling plain Python functions and objects with little ceremony. The unit function needed no changes. It also suits mixed workloads where tasks differ in size, need GPUs or hold state (like the downloader actors). And ML stages can live in the same system: Ray Train for training, Ray Serve for serving.
- **Shuffles and joins:** Ray Data has hash-shuffle group-bys and joins ([Ray Data internals](https://docs.ray.io/en/latest/data/data-internals.html)). They're newer than Spark's. Databricks' own guidance is to use Spark for heavy joins and aggregations and Ray for compute-heavy stages ([Databricks](https://docs.databricks.com/aws/en/machine-learning/ray/spark-ray-overview)).
- **Event time:** Ray has no built-in model of event-time windows, watermarks or late data. Beam and Flink do (notebook 08).
- **Failures:** task retries come from lineage and are cheap if tasks are idempotent. An actor is not restarted by default (`max_restarts=0`). When it is its constructor runs again and any state is lost unless you checkpoint it ([Ray actor fault tolerance](https://docs.ray.io/en/latest/ray-core/fault_tolerance/actors.html)). The head node's control store lives in memory by default so losing the head node loses the cluster. Making it survive needs an external highly available Redis ([Ray GCS fault tolerance](https://docs.ray.io/en/latest/ray-core/fault_tolerance/gcs.html)). That's why the manifest lives on disk and not in an actor.

## What I learned

- **The manifest did more than the framework.** Fingerprinting each input and recording the code version means a rerun with nothing changed does nothing and a changed input is redone alone.
- **Retries are safe because units are idempotent.** Each unit writes atomically so a retry produces the same file and a killed attempt leaves nothing half-written.
- **Processes beat threads by about 3×.** Downloading mostly waits on the network so threads were fine there. Parsing is Python and pandas work that threads can't run in parallel.
- **QA is a different shape of work.** Standardizing needs one station-year. Neighbour checks need every station for the same hours and that regrouping is a shuffle.
- **The head node is a single point of failure.** Ray keeps its control store in memory by default. That's fine for a rerunnable batch job and a real concern for a live pipeline.

**Still open:** where should the manifest live when many machines write to it at once?

In [10]:
ray.shutdown()